# Transformer Blocks
Each transformer block consists of two parts: attention and feedforward. We already previously introduced attention, let's consider feedforward:

## Feedforward
After attention, each token has gathered context from other tokens as its new representation is a weighted sum of value vectors, weighted by attention scores. "Squirrel" now knows about "chased" and "dog". Additionally, attention involved mostly linear operations: while softmax introduces nonlinearity, the core operation-- mixing value vectors-- is a weighted sum. But as I briefly wrote before, there are often non-linear patterns that we will need to extract.  The feedforward block is where each token thinks independently about what it just learned-- no interactions with other tokens. Furthermore, the feedforward blocks involve nonlinear processing, allowing us to extract these nonlinear patterns. Furthermore, the nonlinearization allows us to stack multiple attention and transformer blocks: without nonlinearization, stacking multiple linear transformations is essentially just one big linear transformation -> one big attention layer. The nonlinearity is what makes stacking meaningful as each layer can learn functions the previous couldn't.

### Structure
```
x -> expand -> activate -> compress -> output
where x = attention(x)
```
**1. Expanding**

Not a necessary component of a transformer, but it allows our network more capactity to represent complex functions. Empirically, multiplying embeddings by 4 was found to work well. So we might see:
```
input: 768 -> middle layer: 3072 -> outer layer: 768
output = x @ W1
```
where each of the 3072 inner neurons represents a different nonlinear function fo the input. We rely on the compression layer to select and combine the most useful neurons.

**2. Activating**

This is where we apply nonlinearity to our expanded input. Here are a few functions that could be used to activate:
* ReLU(x) = $max(0, x)$

* GELU(x) = $0.5 * Φ(x) \approx 0.5*x*(1+tanh(\sqrt{\frac{2}{\pi}} * (x+.044715*x^3))$: smooths cutoff near 0, better gradient flow

* SiLU(x) = x * sigmoid(x) = $\frac{x}{1+e^{-x}}$

As full FFN blocks:
```
output = ReLU(W1(x))
output = GELU(W1(x))
output = SiLU(W1(x))*W2(x) (see note)
```
Note that because the input x is of shape [batch, seq_len, embeddings_dim], we need to expand it to hidden_dim before applying the activation (as described in the expanding section). W1 is imply the matrix that does the expansion.

Note: In SwiGLU, W2 is the gate - a second learned projection that decides which values to let through: W1(x) are the values (what information to pass), and W2 acts as the gate (how much of each value to let through). Both matrices are of shape (embeddings_dim, hidden_dim) as they project the same input x into two different 3072-dimensional spaces. Whereas with ReLU and GELU, the gate is set on a fixed rule (>0). Note that neurons that are unable to pass through this gate are dead and stop learning with ReLU. GELU introduced a smoother improvement, and was further improved upon by SiLU.

**3. Compressing**

We just need to get our output back to shape [batch, seq_len, embeddings_dim], and we do so by compressing: ```output = output @ W_compress```
So for the full FFN blocks:
```
output = W2(ReLU(W1(x)))
output = W2(GELU(W1(x)))
output = W3(SiLU(W1(x))*W2(x))
```
## Normalization
Now that we have a basic understanding of attention -> FFN, we can think about additional transformations we must implement as we put them together.
```
Naive approach:
x -> attention(x) -> ffn(x)
slightly more granular:
x -> output = softmax(weighted(x)) -> SwiGLU(output)
```
where
$\sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}} \quad$ and SiLU = $\frac{x}{1+e^{-x}}$

Let's consider an example of 6 transformer layers with no normalization of one token ("dog") with embeddings_dim = 4:
```
layer 1 output: [0.5, 0.3, -0.2, 0.8]
layer 2 output: [1.2, 0.9, -0.6, 1.8]   ← values growing
layer 3 output: [3.1, 2.4, -1.8, 4.2]
layer 4 output: [8.9, 6.7, -5.1, 11.3]
layer 5 output: [24.2, 18.1, -14.2, 30.8]
layer 6 output: [65.7, 49.3, -38.6, 83.7]  ← exploded!

layer 6 dot product: 65*65 = 4225
Softmax[4225, 4224, 4223] = [1, 0, 0] ← peaked, no gradients
```
In makemore, Karpathy implements BatchNorm to avoid the saturating problem in an MLP setting; when inputs to signmoid or tanh are very large or very small, the gradient becomes nearly 0 as the function is asymptotic at the extremes. BatchNorm maintains that the inputs to activation are in a reasonable range so that gradients flow properly. At a high level,  this ensures that training does not break.

$BatchNorm(x) = γ*(\frac{(x-\mu)}{\sigma(x)+ϵ})+β$

where $γ, Β, ϵ$ are learned scale, shifts, and constants to prevent div by 0
respectively.

However, BatchNorm cannot be used here for a few reasons:
1. Variable sequence lengths: different sequences in a batch can have different lengths. Computing batch stats can be messy when this occures.
2. Autoregressive inference: at inference time, we generate one token at a time, so batch size is often 1. But we cannot compute variance across 1 example, which makes BatchNorm in this case undefined.
3. Position dependence: BatchNorm normalizes each feature across the batch, but is unable to handle our case where position matters.

The solution: LayerNorm/RMSNorm, which eliminates cross-batch and cross-position mixing. Karpathy implements LayerNorm in nanoGPT.
### LayerNorm
$LayerNorm(x) = γ*(\frac{(x-\mu)}{\sigma(x)+ϵ})+β$

Formula is exact same as BatchNorm! But rather than computing mean and var across the **batch** dimension, we compute mean and var across the **embedding** dimension.  This way, we are able to normalize independently of batch size, length, and maintain positional information (does not mix information across positions).

### RMSNorm

$RMSNorm(x) = γ*\frac{x}{\sqrt{\mu(x^2)+ϵ}}$

RMSNorm improves the efficiency of LayerNorm but maintains its flexibility. Removing centering, bias, and standard deviation removes unnecessary compute and can maintain performance while increasing efficiency.

Going back to our earlier example, with RMSNorm after each layer,
```
layer 1 output normalized: [0.5, 0.3, -0.2, 0.8]  → RMS ≈ 0.5 → [1.0, 0.6, -0.4, 1.6]
layer 6 output normalized: always stays at consistent scale
```
Thus, we will normalize before attention and before FFN. I was initially confused as to why we normalize before attention given that we scale prior to softmax, but we are essentially just containing two different explosion risks:
1. dot products explode within one attention layer (scaling solves this)
2. representations explode across 12 layers:
```
x = [65, 49, -38, 83]        ← exploded after many layers
RMSNorm(x) = [1.0, 0.7, -0.6, 1.3]   ← brought back to scale
Q = RMSNorm(x) @ W_Q = [0.8, 0.3]    ← projected
K = RMSNorm(x) @ W_K = [0.6, -0.4]
Q·K = 0.8*0.6 + 0.3*(-0.4) = 0.36   ← still manageable
Q·K / sqrt(2) = 0.25                  ← scaled for softmax stability
```
## Residual Connection

The last thing we need to consider before implementing a transformer block.
```
without residual:
x -> attention -> ffn
```

Let's consider a simple example of a network with just 5 layers:
$\frac{∂L}{∂W_1} = \frac{∂L}{∂W_5} * \frac{∂W_5}{∂W_4} * \frac{∂W_4}{∂W_3} * \frac{∂W_3}{∂W_2} * \frac{∂W_2}{∂W_1}$. If each partial derivative is set to .5 (a reasonable approximation), then $\frac{∂L}{∂W_1} = (\frac{1}{2})^4 = \frac{1}{16}$ - a pretty small learning signal.



> And this is a gross oversimplification. In reality:
* each transformer block has many operations (attention, softmax, FFN) each with their own chain of derivatives (and potentially heads)
* With 12 blocks stacked, the gradient flows back through all 12 blocks
* Each block has multiple weight matrices

So the actual gradient chain is enormously long with 12 heads per block and 12 blocks.

### Solution: Residual Connection
```
with residual:
x -> attention + x -> ffn + x
```
We add the original x back so that when we backprop, we have a direct path back to x since the derivative of f(x) + x WRT x >= 1.

$output = x + f(x)$

$\frac{∂output}{∂x} = 1 + f'(x)$ so even if f'(x) vanishes to .0001, the gradient is still 1.0001.

We implement the residual connection at every transformer block during attention and FFN, which yields 24 residual connections total. This ensures that no matter how deep the network, the gradient highway runs all the way from the output back to the input.

Tying everything together, we will normalize before each sublayer so the values are at a consistent scale going in.


In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [28]:
class RMSNorm(nn.Module):
    def __init__(self, embeddings_dim, eps=1e-5):

        super().__init__()

        # input: [batch, seq_len, embeddings_dim]
        self.gamma = nn.Parameter(torch.ones(embeddings_dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
      # compute denom
      rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
      # multiply
      xrms = x / rms
      # gamma
      gxrms = self.gamma * xrms
      return gxrms

class SwiGLU(nn.Module):
    def __init__(self, embeddings_dim, expand: int = 4):

        super().__init__()

        self.W1 = nn.Linear(embeddings_dim, expand * embeddings_dim, bias=False)
        self.W2 = nn.Linear(embeddings_dim, expand * embeddings_dim, bias=False)
        self.W3 = nn.Linear(expand * embeddings_dim, embeddings_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # output = W3(SiLU(W1(x))*W2(x))
        # SiLU = x * sigmoid(x)
        expand = self.W1(x)
        gate = self.W2(x)

        # silu = lambda x : x * torch.sigmoid(x)
        return self.W3(F.silu(expand) * gate)

In [29]:
rms_norm = RMSNorm(4)
x = torch.randn(1,1,2,4)
print(x)
rms_norm(x)

tensor([[[[-0.3723,  2.4145, -0.1717,  0.6740],
          [ 0.7187, -0.8146,  0.0540, -0.4354]]]])


tensor([[[[-0.2931,  1.9011, -0.1352,  0.5307],
          [ 1.2268, -1.3906,  0.0922, -0.7432]]]], grad_fn=<MulBackward0>)

In [30]:
swiglu = SwiGLU(4)
swiglu(x)

tensor([[[[-0.1493,  0.0510,  0.1601,  0.2514],
          [-0.0106,  0.0063, -0.0108,  0.0376]]]],
       grad_fn=<UnsafeViewBackward0>)

In [37]:
# copying over RoPE code
class RoPE(nn.Module):
  def __init__(self, embeddings_dim, max_seq_len: int = 2048, theta: float = 10000.):
    super().__init__()

    assert(embeddings_dim % 2 == 0)

    pair_number = torch.arange(0, (embeddings_dim // 2), 1).float()

    denom = theta ** (2*pair_number / embeddings_dim)
    inv_freq = 1. / denom

    positions = torch.arange(0, max_seq_len, 1)

    pair_angles = torch.outer(positions, inv_freq)

    angles = torch.repeat_interleave(pair_angles, repeats=2, dim=-1)

    sine = torch.sin(angles)
    cosine = torch.cos(angles)
    self.register_buffer("sine_cached", sine)
    self.register_buffer("cos_cached", cosine)

  @staticmethod
  def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    rotated = torch.stack((-x_odd, x_even), dim=-1).flatten(start_dim=-2)
    return rotated

  def forward(self, x: torch.Tensor) -> torch.Tensor:

    seq_len = x.shape[-2]

    cos = self.cos_cached[:seq_len]
    sin = self.sine_cached[:seq_len]

    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)

    x_prime = x * cos + self.rotate_half(x) * sin
    return x_prime

In [36]:
# copying over attention code
class MultiHeadAttention(nn.Module):
  def __init__(self, embeddings_dim: int, head_count: int, dropout: float = .1):
    super().__init__()
    assert embeddings_dim % head_count == 0, "embeddings_dim must be divisible by head_count"
    dim_per_head = embeddings_dim // head_count

    self.head_count = head_count
    self.dim_per_head = dim_per_head
    self.dropout = dropout

    self.w_qkv = nn.Linear(embeddings_dim, 3*embeddings_dim, bias = False)
    self.w_o = nn.Linear(embeddings_dim, embeddings_dim, bias = False)
    self.rope = RoPE(dim_per_head)
    self.attn_dropout = nn.Dropout(dropout)
    self.resid_dropout = nn.Dropout(dropout)

  def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:

    batch_size, seq_len, embeddings_dim = x.shape
    qkv = self.w_qkv(x)

    qkv_reshape = qkv.reshape(batch_size, seq_len, 3, self.head_count, self.dim_per_head)
    qkv = qkv_reshape.permute(2, 0, 3, 1, 4) # [3, batch_size, head_count, seq_len, dim_per_head]
    q, k, v = qkv[0], qkv[1], qkv[2]
    q = self.rope(q)
    k = self.rope(k)
    scores = (q @k.transpose(-1, -2))
    scores_scaled = scores / math.sqrt(self.dim_per_head)
    if mask is not None:
      scores_scaled = scores_scaled.masked_fill(mask == 0, float('-inf'))
    softmax_weights = F.softmax(scores_scaled, dim=-1)
    softmax_weights_dropout = self.attn_dropout(softmax_weights)
    output = softmax_weights_dropout @ v

    output = output.transpose(1, 2).reshape(batch_size, seq_len, embeddings_dim)

    output = self.w_o(output)

    output = self.resid_dropout(output)

    return output

  def make_mask(self, seq_len) -> torch.Tensor:
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)

In [33]:
class TransformerBlock(nn.Module):

  def __init__(self, embeddings_dim, head_count, dropout: int = 0.1):
    super().__init__()
    self.rms1 = RMSNorm(embeddings_dim)
    self.rms2 = RMSNorm(embeddings_dim)
    self.attention = MultiHeadAttention(embeddings_dim, head_count, dropout)
    self.ffn = SwiGLU(embeddings_dim)

  def forward(self, x: torch.Tensor, mask : torch.Tensor = None) -> torch.Tensor:
    x = x + self.attention(self.rms1(x), mask)
    x = x + self.ffn(self.rms2(x))
    return x

In [35]:
transformer = TransformerBlock(8, 2)
x = torch.randn(1,1,8)
transformer(x).shape

torch.Size([1, 1, 8])